</br>
<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>Práctica 1: Análisis, procesamiento y validación
        </h2>
        <h1 style="margin-top: -10px;">
            Pipelines en scikit-learn
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Minería de Datos, 2026-27
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="font-size: 1em; color: #D0D0D0;">
                        José Antonio Gámez Martín
                    </span>
                    <a href="mailto:Jose.Gamez@uclm.es" style="text-decoration: none; color: #888888; font-size: 0.8em; padding-left: 15px;">
                        ✉ Jose.Gamez@uclm.es
                    </a>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="font-size: 1em; color: #D0D0D0;">
                        Pablo Torrijos Arenas
                    </span>
                    <a href="mailto:Pablo.Torrijos@uclm.es" style="text-decoration: none; color: #888888; font-size: 0.8em; padding-left: 15px;">
                        ✉ Pablo.Torrijos@uclm.es
                    </a>
                </div>
                </div>
            </div>
        </div>
    </div>
</div>

En este tutorial vamos a preprocesar un conjunto de datos utilizando `Pipelines` de *scikit-learn*. Vamos a trabajar con el siguiente conjunto de datos: [Student Performance](https://archive.ics.uci.edu/ml/datasets/student+performance), aunque en la versión que os proporcionamos hemos añadido la variable `target` y borrado `[G1,G2,G3]`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

In [2]:
# Cargamos los datos
df = pd.read_csv('student-por.csv')
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,target
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,4,1
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,2,1
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,0,yes,no,no,no,yes,yes,yes,no,4,3,2,2,3,3,6,1
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,no,yes,yes,yes,yes,yes,3,2,2,1,1,5,0,1
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,no,no,yes,yes,no,no,4,3,2,1,2,5,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,MS,F,19,R,GT3,T,2,3,services,other,course,mother,1,3,1,no,no,no,yes,no,yes,yes,no,5,4,2,1,2,5,4,1
645,MS,F,18,U,LE3,T,3,1,teacher,services,course,mother,1,2,0,no,yes,no,no,yes,yes,yes,no,4,3,4,1,1,1,4,1
646,MS,F,18,U,GT3,T,1,1,other,other,course,mother,2,2,0,no,no,no,yes,yes,yes,no,no,1,1,1,1,1,5,6,0
647,MS,M,17,U,LE3,T,3,1,services,services,course,mother,2,1,0,no,no,no,no,no,yes,yes,no,2,4,5,3,4,2,6,1


In [3]:
# Dividimos en variables predictoras y variable objetivo
X = df.drop(columns=["target"])
y = df["target"]

<div style="border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: 10px">

## 1. Pipelines

Los Pipelines son estructuras que permiten encadenar pasos de preprocesamiento y modelado de forma ordenada, reproducible y segura.

Cada paso del flujo de trabajo de Machine Learning (limpieza, escalado, codificación, entrenamiento) se puede definir como un **bloque** dentro de un pipeline.
El objetivo es que todo el proceso sea ejecutable con una sola llamada a `.fit()` o `.predict()`.

Ventajas principales:
- Reproducibilidad del proceso completo.
- Evita fugas de datos entre entrenamiento y validación.
- Facilita la comparación de modelos.
- Mejora la legibilidad y la trazabilidad del código.

En esta práctica veremos desde los pipelines más simples hasta estructuras avanzadas con transformadores personalizados.

### Identificación de tipos de variables

Para crear un Pipeline necesitamos diferenciar las variables **numéricas** y **categóricas**, ya que cada tipo requiere un tratamiento diferente durante el preprocesamiento. 

Vamos a suponer que en el análisis exploratorio previo hemos comprobado que las columnas están correctas y que los tipos de datos han sido cargados adecuadamente.

In [4]:
# Identificar columnas numéricas y categóricas
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Variables numéricas:", num_cols)
print("Variables categóricas:", cat_cols)

Variables numéricas: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Variables categóricas: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


### Pipeline más simple: solo el modelo

Comenzaremos con el Pipeline más básico posible: un modelo de **árbol de decisión** sin preprocesamiento.
Este ejemplo nos servirá para entender la estructura mínima de un pipeline en *scikit-learn*.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# División del conjunto de datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Definimos el pipeline más simple: un único paso con el modelo
pipe_simple = Pipeline([
    ("clf", DecisionTreeClassifier(random_state=42))
])

pipe_simple

Pipeline(steps=[('clf', DecisionTreeClassifier(random_state=42))])

In [6]:
# Entrenamiento
pipe_simple.fit(X_train, y_train)

# Predicciones
y_pred = pipe_simple.predict(X_test)

# Evaluación simple
print(f"Accuracy del Pipeline simple: {accuracy_score(y_test, y_pred):.3f}")

ValueError: could not convert string to float: 'GP'

Este pipeline falla porque el modelo no puede interpretar texto. Vamos a solucionarlo con un bloque de preprocesamiento.

### Pipeline con preprocesamiento

En la celda anterior el modelo falló porque el árbol de decisión no puede trabajar directamente con variables de tipo texto.  
Por eso necesitamos incluir un bloque de **preprocesamiento** dentro del Pipeline.

Este bloque prepara los datos antes de llegar al modelo y se encarga de aplicar distintos tratamientos según el tipo de variable:

- **Imputación de valores faltantes (`SimpleImputer`)**:  
  Sustituye los valores ausentes por un valor calculado o constante. Algunas estrategias comunes son:
  - `"mean"`: reemplazar por la media de la columna (numéricas).  
  - `"median"`: reemplazar por la mediana (robusta frente a valores atípicos).  
  - `"most_frequent"`: reemplazar por el valor más repetido (útil en categóricas).  
  - `"constant"`: asignar un valor fijo, por ejemplo `"missing"` o `0`.

- **Escalado (`StandardScaler`)**:  
  Ajusta las variables numéricas para que tengan media 0 y desviación típica 1. Esto homogeniza magnitudes y mejora la convergencia de algunos modelos. Otras alternativas son:
  - `MinMaxScaler`: reescala cada variable al rango [0, 1].  
  - `RobustScaler`: usa la mediana y el rango intercuartílico, menos sensible a outliers.  
  - `Normalizer`: normaliza cada muestra (fila) en lugar de cada característica.

- **Codificación (`OneHotEncoder`)**:  
  Convierte las variables categóricas en variables numéricas binarias (0/1), creando una columna para cada categoría distinta. Así podemos usar modelos de `scikit-learn`, que de momento tiene muy poca compatibilidad con variables categóricas.

Para aplicar distintos procesos a columnas diferentes, utilizamos `ColumnTransformer`, que combina varios sub-pipelines en un único flujo. El resultado será un único Pipeline que realiza el preprocesamiento completo seguido del modelo final.

In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Pipeline para columnas numéricas
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline para columnas categóricas
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combinamos ambos en un ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Pipeline completo: preprocesamiento + modelo
pipe_full = Pipeline([
    ("pre", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])

# Entrenamiento
pipe_full.fit(X_train, y_train)

# Predicciones
y_pred = pipe_full.predict(X_test)

# Evaluación
print("Resultados en el conjunto de prueba:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
print(f"Precisión: {precision_score(y_test, y_pred):.3f}")
print(f"Sensibilidad (recall): {recall_score(y_test, y_pred):.3f}")

Resultados en el conjunto de prueba:
Accuracy:  0.785
F1-score:  0.875
Precisión: 0.860
Sensibilidad (recall): 0.891


Podemos imprimir el `Pipeline` para ver gráficamente cada parte del mismo:

In [8]:
pipe_full

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'Medu', 'Fedu',
                                                   'traveltime', 'studytime',
                                                   'failures', 'famrel',
                                                   'freetime', 'goout', 'Dalc',
                                                   'Walc', 'health',
                                                   'absences']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['school', 'sex', 'address',
                                                   'famsize', 'Pstatus', 'Mjob',
                                                   'Fjob', 'reason', 'guardian',
                                                   'schoolsup', 'famsup',
                                                   'paid', 'activities',
                                                   'nursery', 'higher',
                                                   'internet', 'romantic'])])),
                ('clf', DecisionTreeClassifier(random_state=42))])

### Aplicar el preprocesamiento por separado

Una ventaja importante de los Pipelines es que cada paso puede ejecutarse y analizarse de forma independiente.  
Esto permite inspeccionar qué ocurre dentro de cada etapa sin necesidad de volver a entrenar todo el pipeline completo.

Podemos acceder a cualquier bloque interno usando `named_steps["nombre_del_paso"]`.  
Por ejemplo:
- `pipe_full.named_steps["pre"]` devuelve el **ColumnTransformer** que aplica el preprocesamiento.
- `pipe_full.named_steps["clf"]` devuelve el **modelo final** (en este caso, el árbol de decisión).

De esta manera, es posible transformar los datos manualmente solo hasta cierta fase, por ejemplo, para ver cómo quedan tras el preprocesamiento antes de pasar al modelo.

In [9]:
# Aplicar solo el bloque de preprocesamiento
X_train_transformed = pipe_full.named_steps["pre"].transform(X_train)

print("Tamaño original:", X_train.shape)
print("Tamaño tras preprocesamiento:", X_train_transformed.shape)

X_train_transformed

Tamaño original: (519, 30)
Tamaño tras preprocesamiento: (519, 56)


array([[-0.59105232,  1.31593641,  1.53552629, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.21197426, -1.32101725, -1.16290987, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.21197426,  0.43695186, -0.26343115, ...,  0.        ,
         0.        ,  1.        ],
       ...,
       [ 1.01500084,  1.31593641,  0.63604757, ...,  1.        ,
         1.        ,  0.        ],
       [-0.59105232, -0.44203269, -0.26343115, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.21197426, -2.2000018 , -0.26343115, ...,  1.        ,
         1.        ,  0.        ]], shape=(519, 56))

In [10]:
# Acceder al modelo dentro del pipeline
pipe_full.named_steps["clf"]

DecisionTreeClassifier(random_state=42)

<div style="border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: 10px">

## 2. Transformadores personalizados

Hasta ahora hemos usado transformadores predefinidos (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`).  
Sin embargo, en muchos proyectos necesitamos aplicar transformaciones específicas que no existen en *scikit-learn*.

Podemos hacerlo de dos formas:

1. **Usando `FunctionTransformer`**, que nos permite integrar una función cualquiera en un pipeline.  
2. **Definiendo un transformador propio**, implementado como clase que hereda de `BaseEstimator` y `TransformerMixin`.

A continuación veremos ambos casos.


### 2.1. `FunctionTransformer`

Vamos a crear un transformador simple que **discretice una variable numérica continua**, por ejemplo, la edad del estudiante.

El objetivo es convertir la edad en categorías ("Joven", "Adulto", "Mayor") para ilustrar cómo una función personalizada puede integrarse en el pipeline.


In [11]:
from sklearn.preprocessing import FunctionTransformer

def discretize_age(df):
    df = df.copy()
    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[0, 17, 20, np.inf],
            labels=["Adolescente", "Joven", "Adulto"]
        )
        df = df.drop(columns=["age"])
    return df

# Crear el FunctionTransformer
age_discretizer = FunctionTransformer(discretize_age)

# Aplicar al conjunto de entrenamiento para ver el resultado
X_transf = age_discretizer.fit_transform(X_train)
X_transf

,school,sex,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,age_group
151,GP,M,U,GT3,T,4,4,services,services,course,mother,1,3,0,no,yes,no,yes,yes,yes,yes,no,5,3,3,1,3,5,0,Adolescente
235,GP,F,U,GT3,T,1,1,at_home,other,reputation,mother,1,3,0,no,yes,no,yes,yes,yes,no,yes,4,3,4,1,1,5,12,Adolescente
431,MS,F,R,GT3,T,3,2,at_home,other,course,father,1,2,1,no,no,no,no,yes,yes,no,yes,4,5,4,1,2,5,0,Adolescente
68,GP,F,R,LE3,T,2,2,health,services,reputation,mother,2,2,0,yes,yes,no,no,yes,yes,yes,no,4,1,3,1,3,4,0,Adolescente
49,GP,F,U,GT3,T,4,4,services,teacher,other,father,1,2,0,yes,yes,no,yes,no,yes,yes,no,4,4,4,1,1,3,2,Adolescente
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,GP,F,U,GT3,T,3,3,at_home,other,course,father,1,2,0,no,yes,no,no,yes,yes,yes,no,4,1,4,1,1,3,8,Joven
301,GP,M,U,GT3,T,4,3,teacher,other,course,mother,1,2,0,no,yes,no,no,no,yes,yes,no,4,3,2,1,1,3,2,Joven
323,GP,M,U,LE3,T,4,3,teacher,services,course,mother,2,1,0,no,no,no,yes,yes,yes,yes,no,4,2,3,1,2,1,0,Joven
467,MS,F,R,GT3,T,2,2,other,services,course,father,3,2,0,no,yes,no,yes,yes,yes,yes,no,5,3,4,1,1,2,1,Adolescente


#### Integrar el transformador en el Pipeline

Podemos incluir este `FunctionTransformer` como un paso adicional dentro del pipeline,
antes del bloque de preprocesamiento general.

De esta forma, el flujo de datos quedará así:
1. Aplicar la función personalizada (`discretize_age`).
2. Preprocesar los datos (imputación, escalado, codificación).
3. Entrenar el modelo.


In [12]:
pipe_custom = Pipeline([
    ("discretize", age_discretizer),
    ("pre", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=42))
])

pipe_custom.fit(X_train, y_train)
y_pred = pipe_custom.predict(X_test)

print(f"Accuracy con transformador personalizado: {accuracy_score(y_test, y_pred):.3f}")

ValueError: A given column is not a column of the dataframe

#### El problema: el `ColumnTransformer` ya no encuentra sus columnas

La celda anterior falla con `A given column is not a column of the dataframe`.

El motivo es que `preprocessor` se construyó con las listas `num_cols` y
`cat_cols` que calculamos **al principio**, cuando todavía existía `age`. Pero
`discretize_age` elimina esa columna y crea `age_group`, así que cuando le toca
el turno al `ColumnTransformer` las listas que guarda ya no se corresponden con
los datos que le llegan.

Una tentación es hacer que la propia función actualice esas listas
(`num_cols.remove("age")`, `cat_cols.append("age_group")`). **No lo hagáis.**
Funciona en la celda, pero por un motivo accidental: el `ColumnTransformer`
guarda una *referencia* a la misma lista, así que la ve cambiar. En cuanto el
pipeline se clona, que es exactamente lo que hace `cross_val_score` en cada
pliegue, el clon recibe una copia propia de las listas, la modificación no le
llega y todo se rompe. Y como `cross_val_score` captura el error, lo único que
veríais es un `nan` sin explicación.

La solución correcta es no fijar las columnas de antemano:
`make_column_selector` las decide **en el momento del `fit`**, mirando el
dataframe que realmente recibe. Así el pipeline se adapta solo a cualquier
transformación que añada o quite columnas, y no depende de ningún estado
global.

In [13]:
from sklearn.compose import make_column_selector

def discretize_age(df):
    """Discretiza la edad. No toca ninguna variable global."""
    df = df.copy()
    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[0, 17, 20, np.inf],
            labels=["Adolescente", "Joven", "Adulto"]
        )
        df = df.drop(columns=["age"])
    return df

age_discretizer = FunctionTransformer(discretize_age)

# Las columnas se eligen por tipo en el momento del fit, no con una lista fija
preprocessor_auto = ColumnTransformer([
    ("num", num_pipe, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipe, make_column_selector(dtype_exclude=np.number)),
])

pipe_custom = Pipeline([
    ("discretize", age_discretizer),
    ("pre", preprocessor_auto),
    ("clf", DecisionTreeClassifier(random_state=42))
])

pipe_custom.fit(X_train, y_train)
y_pred = pipe_custom.predict(X_test)

print(f"Accuracy con transformador personalizado: {accuracy_score(y_test, y_pred):.3f}")

Accuracy con transformador personalizado: 0.777


Y ahora sí funciona la validación cruzada, que es como vais a evaluar los
modelos en la práctica. Comparad con lo que daría el pipeline anterior: un
`nan` silencioso.

In [14]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe_custom, X, y, cv=5, scoring="f1")
print(f"F1 en validación cruzada: {scores.mean():.3f} (+/- {scores.std():.3f})")

F1 en validación cruzada: 0.863 (+/- 0.064)


## 2.2: Transformador personalizado con clase propia: limpieza de texto en columnas categóricas

En el dataset de estudiantes, algunas variables categóricas (como `Mjob`, `Fjob`, `guardian`) pueden contener pequeñas inconsistencias de formato: mayúsculas, minúsculas, espacios o caracteres especiales.

Vamos a crear un transformador que limpie texto en varias columnas, homogenizando su formato antes del preprocesamiento.


In [18]:
from sklearn.base import BaseEstimator, TransformerMixin

class CleanCategoricalText(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[col] = (
                    X[col]
                    .astype(str)
                    .str.replace(r"[_\-]+", " ", regex=True)   # reemplazar guiones por espacios
                    .str.strip()                               # eliminar espacios iniciales/finales
                    .str.lower()                               # pasar a minúsculas
                    .str.replace(r"[^a-z ]", "", regex=True)   # quitar caracteres no alfabéticos
                    .str.title()                               # primera letra mayúscula en cada palabra
                )
        return X

#### Integrar varios transformadores personalizados en un Pipeline

Ahora combinaremos los dos transformadores personalizados dentro de un único pipeline:

1. `CleanCategoricalText`: limpia las columnas categóricas seleccionadas.  
2. `discretize_age`: discretiza la variable de edad y actualiza las listas de columnas.  
3. `pre`: aplica el preprocesamiento general (imputación, escalado, codificación).  
4. `clf`: entrena el árbol de decisión final.

Esto permite mantener todo el flujo de limpieza, transformación y modelado en una sola estructura reproducible.

In [16]:
# Definimos las columnas que queremos limpiar
cols_to_clean = ["Mjob", "Fjob", "guardian"]

# Creamos el pipeline completo con todos los pasos personalizados
pipe_final = Pipeline([
    ("clean_text", CleanCategoricalText(cols_to_clean)),
    ("discretize", age_discretizer),
    ("pre", preprocessor_auto),
    ("clf", DecisionTreeClassifier(random_state=42))
])

# Entrenamiento y evaluación
pipe_final.fit(X_train, y_train)
y_pred = pipe_final.predict(X_test)

print("Resultados finales del pipeline:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.3f}")
print(f"Precisión: {precision_score(y_test, y_pred):.3f}")
print(f"Sensibilidad (recall): {recall_score(y_test, y_pred):.3f}")

Resultados finales del pipeline:
Accuracy:  0.777
F1-score:  0.871
Precisión: 0.852
Sensibilidad (recall): 0.891


In [17]:
pipe_final

Pipeline(steps=[('clean_text',
                 CleanCategoricalText(columns=['Mjob', 'Fjob', 'guardian'])),
                ('discretize',
                 FunctionTransformer(func=<function discretize_age at 0x000001C416DB1E40>)),
                ('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001C414691550>),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001C416EE5E50>)])),
                ('clf', DecisionTreeClassifier(random_state=42))])